In [1]:
import pandas as pd
import numpy as np
import os
os.makedirs("output", exist_ok=True)
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, RandomizedSearchCV, HalvingRandomSearchCV,cross_val_score, KFold
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score, mean_absolute_percentage_error
from mapie.regression import MapieRegressor
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, kpss
from datetime import timedelta, datetime


import optuna
from shap import TreeExplainer, summary_plot, dependence_plot, Explainer
import json
from scipy.optimize import minimize
from sklearn.base import BaseEstimator, RegressorMixin

import joblib

pd.set_option("display.max_rows", None)

## Data Preprocessing

In [2]:
# Load data
file_path = "input/Ventas_2022-2024_ALL_RC_FINAL.xlsx"
df = pd.read_excel(file_path, sheet_name="Sheet1", parse_dates=["Date"])
file_path_dim = "input/Dimensão_Produto.xlsx"
df_dim = pd.read_excel(file_path_dim, sheet_name="Sheet1")

In [3]:
print(df.columns)
df.info()

Index(['Date', 'ProductId', 'SKU', 'Name', 'Brand', 'Model',
       'Tipo de producto', 'Cod. PLU', 'Quantity', 'Flag Card', 'Brand-Model',
       'Value', 'Price Mademsa', 'Price Rheem', 'Price Splendid',
       'Price_Sindelen', 'Price (UN)', 'Store Name', 'Price_Anwo',
       'Normal Price', 'Flag Return', 'Flag OUT_OF_STOCK'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79638 entries, 0 to 79637
Data columns (total 22 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Date               79638 non-null  datetime64[ns]
 1   ProductId          79638 non-null  int64         
 2   SKU                79638 non-null  object        
 3   Name               79638 non-null  object        
 4   Brand              79638 non-null  object        
 5   Model              79638 non-null  object        
 6   Tipo de producto   79638 non-null  object        
 7   Cod. PLU           79638 non-null  i

In [4]:
#see if there is any missing date in the date range
date_range = pd.date_range(start=df["Date"].min(), end=df["Date"].max())
missing_dates = date_range.difference(df["Date"])
print(missing_dates)

DatetimeIndex([], dtype='datetime64[ns]', freq='D')


In [5]:
#remove duplicates from df_dim based on ProductId
df_dim = df_dim.drop_duplicates(subset=["ProductId"])

#filter df_dim by brand "Bosch", "Junkers" and "Neckar"
#df_dim = df_dim[df_dim["Brand"].isin(["Bosch", "Junkers", "Neckar"])]

# Remove blank ProductId rows
df_dim = df_dim[df_dim["ProductId"].notna() & (df_dim["ProductId"].astype(str).str.strip() != "")]

# merge df with cols Tipo de combustible, Capacidad por minuto, Capacidad por volumen, Calefones Tiro Natural, Capacidad from df_dim on df_dim.ProductId  = df.ProductId
df = df.merge(df_dim[["ProductId", "Tipo de combustible"]], on="ProductId",how="left", validate="many_to_one")
#df.drop("ProductId", axis=1, inplace=True)

In [6]:
#save df to csv in ouput/Thesis folder
df.to_csv("output/Thesis/df_cleaned.csv", index=False)


In [7]:
# new section
df = df[df["Date"] < "2024-12-09"]


#Print start and end date
print(df["Date"].min())
print(df["Date"].max())

2022-09-15 00:00:00
2024-12-08 00:00:00


In [8]:
print(len(df))

79490


In [9]:
# Show Unique values of "Tipo de producto"
print(df["Tipo de producto"].unique())

['Calefont' 'Termo' 'Caldera']


In [10]:
# Filter 'Convector' and 'Aire Acondicionado' products lines
#df = df[df["Tipo de producto"].isin(["Calefont","Termo", "Caldera"])] #new section - filtered Termo


In [11]:
#print(df["Tipo de producto"].unique())

In [12]:
#sum col "value"
print(df["Value"].sum())
#sum col "Quantity"
print(df["Quantity"].sum())
#average Col "Price (UN)"
print(df["Price (UN)"].mean())

#mean and var of "Quantity"
print(df["Quantity"].mean())
print(df["Quantity"].var())

55518181573
254182
415402.1163794188
3.197660083029312
126.49388424530996


In [13]:
df.sort_values(by=["Date"], inplace=True)
#date feature engineering
df["dayYear"] = df["Date"].dt.dayofyear
df["dayWeek"] = df["Date"].dt.day_of_week
df["dayMonth"] = df["Date"].dt.day
df["sin_day"] = np.sin(2 * np.pi * df["dayYear"] / 365)
df["cos_day"] = np.cos(2 * np.pi * df["dayYear"] / 365)
df["isoWeek"] = df["Date"].dt.isocalendar().week.astype(int)
df['week_sin'] = np.sin(2*np.pi*(df['isoWeek']-1)/52)
df['week_cos'] = np.cos(2*np.pi*(df['isoWeek']-1)/52)
df["year"] = df["Date"].dt.year
df["month"] = df["Date"].dt.month

#transform Date to ordinal
df["date_ordinal"] = df["Date"].map(pd.Timestamp.toordinal)

def season_label(date):
    md = (date.month, date.day)
    if (md >= (12,21)) or (md < (3,20)):
        return 1 #summer
    if (md >= (3,20)) and (md < (6,21)):
        return 2 #autumn
    if (md >= (6,21)) and (md < (9,21)):
        return 3 #winter
    return 4 #spring

df["seasons"] = df["Date"].apply(season_label).astype(int)

#df["PLU_Label"] = LabelEncoder().fit_transform(df["Cod. PLU"])
df["Seller_Code"] = df["Store Name"].map({"Easy": 0, "Sodimac": 1}).fillna(-1).astype(int)

#Create Product Code
df["Tipo_Producto_Code"] = df["Tipo de producto"].map({
    "Calefont": 0,
    "Termo": 1,
    "Caldera": 2
})

#Create Tipo de combustible code
df["Tipo_Combustible_Code"] = df["Tipo de combustible"].map({
    "Gas Natural": 0,
    "Gas Licuado": 1,
    "Eléctrico": 2,
})


In [14]:

# Black Friday dates
bf_dates = pd.to_datetime([
    "2022-11-25",
    "2023-11-24",
    #"2024-11-29"
])

#Cyber Monday (BF + 3 days)
cm_dates = bf_dates + pd.Timedelta(days=0)

# CyberDay Chile
cyberday_dates = pd.to_datetime([
    "2023-05-29",
    "2024-06-03"
])

# Flags (single day)
df["flag_black_friday"] = df["Date"].isin(bf_dates).astype(int)
df["flag_cyber_monday"] = df["Date"].isin(cm_dates).astype(int)
df["flag_cyberday"] = df["Date"].isin(cyberday_dates).astype(int)

# Windows
def create_window_flag(dates, before=0, after=3):
    return (
        df["Date"].isin(dates) |
        df["Date"].isin(dates - pd.Timedelta(days=before)) |
        df["Date"].isin(dates - pd.Timedelta(days=1)) |
        df["Date"].isin(dates + pd.Timedelta(days=1)) |
        df["Date"].isin(dates + pd.Timedelta(days=after))
    ).astype(int)

df["flag_bf_cm_window"] = create_window_flag(bf_dates) | create_window_flag(cm_dates)
df["flag_cyberday_window"] = create_window_flag(cyberday_dates)

# Final combined flag
df["Flag_major_event"] = (
    df["flag_bf_cm_window"] |
    df["flag_cyberday_window"]
).astype(int)

#drop columns "flag_black_friday", "flag_cyber_monday", "flag_cyberday", flag_bf_cm_window, flag_cyberday_window
df.drop(columns=["flag_black_friday", "flag_cyber_monday", "flag_cyberday", "flag_bf_cm_window", "flag_cyberday_window"], inplace=True)


In [15]:
df["Store Name"].unique()

array(['Sodimac', 'Easy'], dtype=object)

In [16]:
#show Nan values
print(df.isnull().sum())

#drop Lowest Price (UN) column
#df.drop(columns=["Lowest Price (UN)"], inplace=True)

Date                         0
ProductId                    0
SKU                          0
Name                         0
Brand                        0
Model                        0
Tipo de producto             0
Cod. PLU                     0
Quantity                     0
Flag Card                    0
Brand-Model                  0
Value                        0
Price Mademsa            63457
Price Rheem              34633
Price Splendid           47523
Price_Sindelen           75298
Price (UN)                   0
Store Name                   0
Price_Anwo               71125
Normal Price             22145
Flag Return                  0
Flag OUT_OF_STOCK            0
Tipo de combustible          0
dayYear                      0
dayWeek                      0
dayMonth                     0
sin_day                      0
cos_day                      0
isoWeek                      0
week_sin                     0
week_cos                     0
year                         0
month   

In [17]:
# new section
df = df.groupby("Cod. PLU").filter(lambda x: len(x) >= 100)

#if Quantity col < 0 then set row to 0
df.loc[df["Quantity"] < 0, "Quantity"] = 0

In [18]:
print(len(df))

79226


In [19]:
lag_days = 7

# Pre-group once
group_qty = df.groupby(["Cod. PLU", "Seller_Code"])["Quantity"]
group_price = df.groupby(["Cod. PLU", "Seller_Code"])["Price (UN)"]
group_value = df.groupby(["Cod. PLU", "Seller_Code"])["Value"]

# Build lag features efficiently
quantity_lags = pd.concat(
    [group_qty.shift(lag).rename(f"quantity_lag_{lag}") for lag in range(1, lag_days + 1)],
    axis=1
)
price_lags = pd.concat(
    [group_price.shift(lag).rename(f"price_lag_{lag}") for lag in range(1, lag_days + 1)],
    axis=1
)

value_lags = pd.concat(
    [group_value.shift(lag).rename(f"value_lag_{lag}") for lag in range(1, lag_days + 1)],
    axis=1
)

# Add all at once
df = pd.concat([df, quantity_lags, price_lags, value_lags], axis=1)

In [20]:
#window days
rolling_SMA50_days = 50
rolling_SMA20_days = 20

g = df.groupby(["Cod. PLU", "Seller_Code"], sort=False)

# SMA - Simple Moving Average
df["rolling_SMA50_price"] = g["Price (UN)"].transform(lambda x: x.shift(1).rolling(window=rolling_SMA50_days, min_periods=1).mean())
df["rolling_SMA50_quantity"] = g["Quantity"].transform(lambda x: x.shift(1).rolling(window=rolling_SMA50_days, min_periods=1).mean())
df["rolling_SMA50_value"] = df["rolling_SMA50_price"] * df["rolling_SMA50_quantity"]

df["rolling_SMA20_price"] = g["Price (UN)"].transform(lambda x: x.shift(1).rolling(window=rolling_SMA20_days, min_periods=1).mean())
df["rolling_SMA20_quantity"] = g["Quantity"].transform(lambda x: x.shift(1).rolling(window=rolling_SMA20_days, min_periods=1).mean())
df["rolling_SMA20_value"] = df["rolling_SMA20_price"] * df["rolling_SMA20_quantity"]


#df.dropna(inplace=True)
#df.fillna(0, inplace=True)

In [21]:
rolling_days_EMA = 7
# EMA - Exponential Moving Average
def calculate_ema(series, span):
    return series.ewm(span=span, adjust=False).mean()
# Calculate EMA for Price and Quantity

df["rolling_EMA_price"] = df.groupby(["Cod. PLU", "Seller_Code"])["Price (UN)"].transform(lambda x: calculate_ema(x.shift(1), span=rolling_days_EMA))
df["rolling_EMA_quantity"] = df.groupby(["Cod. PLU", "Seller_Code"])["Quantity"].transform(lambda x: calculate_ema(x.shift(1), span=rolling_days_EMA))
df["rolling_EMA_value"] = df["rolling_EMA_price"] * df["rolling_EMA_quantity"]

In [22]:
df['Normal Price'] = df.groupby(["Cod. PLU", "Seller_Code"])['Normal Price'].transform('ffill')
if df["Normal Price"].isnull().any():
    df['Normal Price'] = df.groupby(["Cod. PLU"])['Normal Price'].transform(lambda x: x.ffill().bfill())
    
df["price_diff"] = df["Normal Price"] - df["Price (UN)"]
df["discount_price"] = df["price_diff"] / df["Normal Price"] #.replace(0, np.nan)  # Avoid division by zero
df["discount_price"] = df["discount_price"].clip(0, 1)
df["Flag Promo"] = (df["Price (UN)"] < df["Normal Price"]).astype(int)

In [23]:
#price vs competitor
competitor_cols = ["Price Splendid", "Price Rheem", 'Price Mademsa', 'Price_Sindelen', 'Price_Anwo']

df["min_comp_price"] = df[competitor_cols].min(axis=1, skipna=True)
df["Flag_competitor_price"] = df[competitor_cols].notna().any(axis=1).astype(int)


#price diff competitor
df["price_diff_competitor"] =  df["min_comp_price"] -df["Price (UN)"]
#discount price vs competitor
df["discount_price_vs_competitor"] = df["price_diff_competitor"] / df["min_comp_price"]
df["discount_price_vs_competitor"] = df["discount_price_vs_competitor"].clip(-1, 1) #.fillna(0)
df["Flag_discount_price_vs_competitor"] = (df["Price (UN)"] < df["min_comp_price"]).astype(int)

In [24]:
df.info()
#df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 79226 entries, 0 to 70198
Data columns (total 77 columns):
 #   Column                             Non-Null Count  Dtype         
---  ------                             --------------  -----         
 0   Date                               79226 non-null  datetime64[ns]
 1   ProductId                          79226 non-null  int64         
 2   SKU                                79226 non-null  object        
 3   Name                               79226 non-null  object        
 4   Brand                              79226 non-null  object        
 5   Model                              79226 non-null  object        
 6   Tipo de producto                   79226 non-null  object        
 7   Cod. PLU                           79226 non-null  int64         
 8   Quantity                           79226 non-null  int64         
 9   Flag Card                          79226 non-null  int64         
 10  Brand-Model                        7922

In [42]:
#save df to with enginnered features to csv in ouput/Thesis folder
df.to_csv("output/Thesis/df_cleaned_engineered.csv", index=False)

## Train XGBoost Monotone model and save it to file to use in Streamlit

In [25]:
def train_model_XGB_M(df):

    features = ["Cod. PLU","Seller_Code", "Price (UN)","Tipo_Combustible_Code","date_ordinal", # Categorical features
                    "year", "seasons","month", "week_sin", "week_cos", "sin_day", "cos_day", #Time features
                    "discount_price_vs_competitor", "Flag_competitor_price","discount_price","Flag Promo", "Flag Card", "Flag_major_event", "Flag Return",#competition and promo features
                    "rolling_SMA50_price", "rolling_SMA50_quantity",
                    "rolling_SMA20_price", "rolling_SMA20_quantity", 
                    "rolling_EMA_price", "rolling_EMA_quantity",
                    ] + [f"quantity_lag_{i}" for i in range(1, lag_days + 1)] + [f"price_lag_{i}" for i in range(1, lag_days + 1)] 



    target = "Quantity"

    #test_size = 0.1

    # Split off a test set
    #X_train, X_test, y_train, y_test = train_test_split(
    #    X, y, test_size=test_size, shuffle=False
    #)

    #full dataset
    X = df[features]
    y = df[target]

    split_date = "2024-10-13"

    X_train = df.loc[df["Date"] <= split_date, features]
    X_test  = df.loc[df["Date"] > split_date, features]

    y_train = df.loc[df["Date"] <= split_date, target]
    y_test  = df.loc[df["Date"] > split_date, target]



    # 3) List out exactly which features should be constrained
    # define your two sets
    dec_features = {
        "Price (UN)",
        "rolling_SMA50_price",
        "rolling_SMA20_price",
        "rolling_EMA_price",
        *{f"price_lag_{i}" for i in range(1, lag_days + 1)},
    
        
    }

    inc_features = {
        "discount_price",
        "discount_price_vs_competitor",
        
    }

    # build one monotone tuple over all features
    monotone = tuple(
        -1 if feat in dec_features
        else +1 if feat in inc_features
        else 0
        for feat in features
    )


    model_xgb_m = XGBRegressor(monotone_constraints= monotone, subsample=1, n_estimators=1500, min_child_weight=1, max_depth=3, reg_lambda = 3, learning_rate=0.01,
                            gamma=0.7, colsample_bytree=0.8, random_state=42, objective='reg:squarederror') #**xgb_cv.best_params_

    model_xgb_m.fit(X_train, y_train)
   
    # Predict on TEST
    y_pred = model_xgb_m.predict(X_test)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    print(f"MAE: {mae:.4f}, RMSE: {rmse:.4f}, R²: {r2:.4f}")

    #selector = SelectFromModel(model, threshold="median", prefit=True)  # or use a float threshold
    #selected_features = X_train.columns[selector.get_support()]

    #print(f"Selected Features ({len(selected_features)}):")
    #print(selected_features.tolist())

    #Refit the model on the full dataset for production use after evaluating on the test split
    model_xgb_m.fit(X, y)

    conformal_xgb_m = MapieRegressor(model_xgb_m, method="plus", cv="prefit") #alternative "naive"
    conformal_xgb_m.fit(X, y)

    metrics = {
        "mae": mae,
        "rmse": rmse,
        "r2": r2
    }

    with open("models/metrics_xgb_m.json", "w") as f:
        json.dump(metrics, f, indent=4)

    return model_xgb_m, conformal_xgb_m

In [26]:
model_xgb_m, conformal_xgb_m = train_model_XGB_M(df)

MAE: 1.2500, RMSE: 3.0126, R²: 0.8401


In [27]:
joblib.dump(
    {"model": model_xgb_m, "mapie": conformal_xgb_m},
    "models/model_and_mapie_xgb_m.joblib"
)

['models/model_and_mapie_xgb_m.joblib']

In [28]:
#save model and conformal predictor in same file
#joblib.dump((model_xgb, conformal_xgb), 'model_and_mapie_xgb.joblib')

## Train XGBoost model and save it to file to use in Streamlit

In [29]:
def train_model_XGB(df):

    features = ["Cod. PLU","Seller_Code", "Price (UN)","Tipo_Combustible_Code","date_ordinal", # Categorical features
                    "year", "seasons","month", "week_sin", "week_cos", "sin_day", "cos_day", #Time features
                    "discount_price_vs_competitor", "Flag_competitor_price","discount_price","Flag Promo", "Flag Card", "Flag_major_event", "Flag Return",#competition and promo features
                    "rolling_SMA50_price", "rolling_SMA50_quantity",
                    "rolling_SMA20_price", "rolling_SMA20_quantity", 
                    "rolling_EMA_price", "rolling_EMA_quantity",
                    ] + [f"quantity_lag_{i}" for i in range(1, lag_days + 1)] + [f"price_lag_{i}" for i in range(1, lag_days + 1)] 



    target = "Quantity"
    #test_size = 0.1

    #Split off a test set
    #X_train, X_test, y_train, y_test = train_test_split(
    #    X, y, test_size=test_size, shuffle=False
    #)

    #full dataset
    X = df[features]
    y = df[target]


    split_date = "2024-10-13"

    X_train = df.loc[df["Date"] <= split_date, features]
    X_test  = df.loc[df["Date"] > split_date, features]

    y_train = df.loc[df["Date"] <= split_date, target]
    y_test  = df.loc[df["Date"] > split_date, target]


    model_xgb = XGBRegressor(subsample=1, n_estimators=1500, min_child_weight=1, max_depth=3, reg_lambda = 3, learning_rate=0.01, #best params = max_depth = 6
                            gamma=0.7, colsample_bytree=0.8, random_state=42, objective='reg:squarederror') #**xgb_cv.best_params_

    model_xgb.fit(X_train, y_train)
   
    # Predict on TEST
    y_pred = model_xgb.predict(X_test)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    print(f"MAE: {mae:.4f}, RMSE: {rmse:.4f}, R²: {r2:.4f}")

    #selector = SelectFromModel(model, threshold="median", prefit=True)  # or use a float threshold
    #selected_features = X_train.columns[selector.get_support()]

    #print(f"Selected Features ({len(selected_features)}):")
    #print(selected_features.tolist())

    #Refit the model on the full dataset for production use after evaluating on the test split
    model_xgb.fit(X, y)

    conformal_xgb = MapieRegressor(model_xgb, method="plus", cv="prefit") #alternative "naive"
    conformal_xgb.fit(X, y)

    metrics = {
        "mae": mae,
        "rmse": rmse,
        "r2": r2
    }

    with open("models/metrics_xgb.json", "w") as f:
        json.dump(metrics, f, indent=4)

    return model_xgb, conformal_xgb

In [30]:
model_xgb, conformal_xgb = train_model_XGB(df)

MAE: 1.2571, RMSE: 2.9839, R²: 0.8431


In [31]:
joblib.dump(
    {"model": model_xgb, "mapie": conformal_xgb},
    "models/model_and_mapie_XGB.joblib"
)

['models/model_and_mapie_XGB.joblib']

## Train Catboost Monotone model and save it to file to use in Streamlit

In [32]:
def train_model_CAT_M(df):

    features = ["Cod. PLU","Seller_Code", "Price (UN)","Tipo_Combustible_Code","date_ordinal", # Categorical features
                    "year", "seasons","month", "week_sin", "week_cos", "sin_day", "cos_day", #Time features
                    "discount_price_vs_competitor", "Flag_competitor_price","discount_price","Flag Promo", "Flag Card", "Flag_major_event", "Flag Return",#competition and promo features
                    "rolling_SMA50_price", "rolling_SMA50_quantity",
                    "rolling_SMA20_price", "rolling_SMA20_quantity", 
                    "rolling_EMA_price", "rolling_EMA_quantity",
                    ] + [f"quantity_lag_{i}" for i in range(1, lag_days + 1)] + [f"price_lag_{i}" for i in range(1, lag_days + 1)] 



    target = "Quantity"
    #test_size = 0.1

    #Split off a test set
    #X_train, X_test, y_train, y_test = train_test_split(
    #    X, y, test_size=test_size, shuffle=False
    #)

    #full dataset
    X = df[features]
    y = df[target]


    split_date = "2024-10-13"

    X_train = df.loc[df["Date"] <= split_date, features]
    X_test  = df.loc[df["Date"] > split_date, features]

    y_train = df.loc[df["Date"] <= split_date, target]
    y_test  = df.loc[df["Date"] > split_date, target]

    dec_features = {
        "Price (UN)",
        "rolling_SMA50_price",
        "rolling_SMA20_price",
        "rolling_EMA_price",
        *{f"price_lag_{i}" for i in range(1, lag_days + 1)},
    
        
    }

    inc_features = {
        "discount_price",
        "discount_price_vs_competitor",
        #"Flag_major_event",
        #"Flag Promo",
        #"Flag_card",
        
    }

    catboost_monotone = [
        -1 if feat in dec_features
        else +1 if feat in inc_features
        else 0
        for feat in features
    ]


    model_cat_m = CatBoostRegressor(monotone_constraints=catboost_monotone, n_estimators=1500, learning_rate=0.01, depth=6, l2_leaf_reg=3, min_data_in_leaf=1, subsample=0.6,
                            rsm=0.6, loss_function="RMSE", eval_metric="RMSE", random_seed=42, verbose=0)
    model_cat_m.fit(X_train, y_train)
   
    # Predict on TEST
    y_pred = model_cat_m.predict(X_test)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    print(f"MAE: {mae:.4f}, RMSE: {rmse:.4f}, R²: {r2:.4f}")

    #selector = SelectFromModel(model, threshold="median", prefit=True)  # or use a float threshold
    #selected_features = X_train.columns[selector.get_support()]

    #print(f"Selected Features ({len(selected_features)}):")
    #print(selected_features.tolist())

    #Refit the model on the full dataset for production use after evaluating on the test split
    model_cat_m.fit(X, y)

    conformal_cat_m = MapieRegressor(model_cat_m, method="plus", cv="prefit") #alternative "naive"
    conformal_cat_m.fit(X, y)

    metrics = {
        "mae": mae,
        "rmse": rmse,
        "r2": r2
    }

    with open("models/metrics_cat_m.json", "w") as f:
        json.dump(metrics, f, indent=4)

    return model_cat_m, conformal_cat_m

In [33]:
model_cat_m, conformal_cat_m = train_model_CAT_M(df)

MAE: 1.2640, RMSE: 3.0497, R²: 0.8361


In [34]:
joblib.dump(
    {"model": model_cat_m, "mapie": conformal_cat_m},
    "models/model_and_mapie_cat_m.joblib"
)

['models/model_and_mapie_cat_m.joblib']

## Train Linear Regression model and save it to file to use in Streamlit

## NANs

In [35]:
#show Nan values
print(df.isnull().sum())


Date                                     0
ProductId                                0
SKU                                      0
Name                                     0
Brand                                    0
Model                                    0
Tipo de producto                         0
Cod. PLU                                 0
Quantity                                 0
Flag Card                                0
Brand-Model                              0
Value                                    0
Price Mademsa                        63279
Price Rheem                          34443
Price Splendid                       47356
Price_Sindelen                       75034
Price (UN)                               0
Store Name                               0
Price_Anwo                           70935
Normal Price                             0
Flag Return                              0
Flag OUT_OF_STOCK                        0
Tipo de combustible                      0
dayYear    

In [36]:
#make a copy of df
df_copy = df.copy()

In [37]:
#fill df_copy NaN values with 0
df_copy.fillna(0, inplace=True)

In [38]:
#show Nan values
print(df_copy.isnull().sum())

Date                                 0
ProductId                            0
SKU                                  0
Name                                 0
Brand                                0
Model                                0
Tipo de producto                     0
Cod. PLU                             0
Quantity                             0
Flag Card                            0
Brand-Model                          0
Value                                0
Price Mademsa                        0
Price Rheem                          0
Price Splendid                       0
Price_Sindelen                       0
Price (UN)                           0
Store Name                           0
Price_Anwo                           0
Normal Price                         0
Flag Return                          0
Flag OUT_OF_STOCK                    0
Tipo de combustible                  0
dayYear                              0
dayWeek                              0
dayMonth                 

## Model

In [39]:
def train_model_LR(df):

    features = ["Cod. PLU","Seller_Code", "Price (UN)","Tipo_Combustible_Code","date_ordinal", # Categorical features
                    "year", "seasons","month", "week_sin", "week_cos", "sin_day", "cos_day", #Time features
                    "discount_price_vs_competitor", "Flag_competitor_price","discount_price","Flag Promo", "Flag Card", "Flag_major_event", "Flag Return",#competition and promo features
                    "rolling_SMA50_price", "rolling_SMA50_quantity",
                    "rolling_SMA20_price", "rolling_SMA20_quantity", 
                    "rolling_EMA_price", "rolling_EMA_quantity",
                    ] + [f"quantity_lag_{i}" for i in range(1, lag_days + 1)] + [f"price_lag_{i}" for i in range(1, lag_days + 1)] 



    target = "Quantity"
    #test_size = 0.1

    #Split off a test set
    #X_train, X_test, y_train, y_test = train_test_split(
    #    X, y, test_size=test_size, shuffle=False
    #)

    #full dataset
    X = df[features]
    y = df[target]


    split_date = "2024-10-13"

    X_train = df.loc[df["Date"] <= split_date, features]
    X_test  = df.loc[df["Date"] > split_date, features]

    y_train = df.loc[df["Date"] <= split_date, target]
    y_test  = df.loc[df["Date"] > split_date, target]


    model_lr = LinearRegression()
    
    model_lr.fit(X_train, y_train)
   
    # Predict on TEST
    y_pred = model_lr.predict(X_test)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    print(f"MAE: {mae:.4f}, RMSE: {rmse:.4f}, R²: {r2:.4f}")

    #selector = SelectFromModel(model, threshold="median", prefit=True)  # or use a float threshold
    #selected_features = X_train.columns[selector.get_support()]

    #print(f"Selected Features ({len(selected_features)}):")
    #print(selected_features.tolist())

    #Refit the model on the full dataset for production use after evaluating on the test split
    model_lr.fit(X, y)

    conformal_lr = MapieRegressor(model_lr, method="plus", cv="prefit") #alternative "naive"
    conformal_lr.fit(X, y)

    metrics = {
        "mae": mae,
        "rmse": rmse,
        "r2": r2
    }

    with open("models/metrics_lr.json", "w") as f:
        json.dump(metrics, f, indent=4)

    return model_lr, conformal_lr

In [40]:
model_lr, conformal_lr = train_model_LR(df_copy)

MAE: 1.3374, RMSE: 3.0607, R²: 0.8349


In [41]:
joblib.dump(
    {"model": model_lr, "mapie": conformal_lr},
    "models/model_and_mapie_LR.joblib"
)

['models/model_and_mapie_LR.joblib']